<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 1: Chunking 비교

Google Colab 환경에서 **고정 길이 Chunking**과 **문장 경계 기반 Chunking**을 비교합니다.

### 학습 목표
- 문서를 일정한 토큰 길이로 나누는 방법을 이해한다.
- 문장 경계를 보존하는 Chunking을 이해한다.
- 두 방법의 장단점을 비교한다.
- Chunk 크기에 따라 결과가 어떻게 달라지는지 관찰한다.


## 1. 필요한 라이브러리 설치

In [1]:
# Hugging Face tokenizer와 한국어 문장 분리기 설치
!pip -q install transformers kss

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 51.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.8/468.8 kB 18.3 MB/s eta 0:00:00


## 2. 실습용 문서 준비

In [2]:
# 실습에 사용할 한국어 문서
text = """
RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다.
RAG 시스템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다.
이렇게 나눈 작은 문서 조각을 chunk라고 합니다.
Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다.
반대로 Chunk가 너무 작으면 문장의 의미가 잘릴 수 있습니다.
따라서 문서의 특성에 맞는 Chunking 방법을 선택하는 것이 중요합니다.
고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있습니다.
문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.
"""

print(text)


RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다.
RAG 시스템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다.
이렇게 나눈 작은 문서 조각을 chunk라고 합니다.
Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다.
반대로 Chunk가 너무 작으면 문장의 의미가 잘릴 수 있습니다.
따라서 문서의 특성에 맞는 Chunking 방법을 선택하는 것이 중요합니다.
고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있습니다.
문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.



## 3. Tokenizer 준비

Tokenizer는 문장을 모델이 처리할 수 있는 작은 단위(token)로 나눕니다.

In [3]:
from transformers import AutoTokenizer

# 다국어 BERT tokenizer 사용
# 한국어 문장을 token 단위로 나누기 위해 사용합니다.
tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-multilingual-cased"
)

# 전체 문서를 token으로 변환
tokens = tokenizer.tokenize(text)

print("전체 token 수:", len(tokens))
print()
print("앞부분 token:")
print(tokens[:30])

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

전체 token 수: 175

앞부분 token:
['RA', '##G', '##는', '외', '##부', '문서', '##를', '검', '##색', '##하여', 'LL', '##M', '##의', '답', '##변', '생', '##성', '##에', '활', '##용', '##하는', '기', '##술', '##입', '##니다', '.', 'RA', '##G', '시', '##스']


# 방법 1. 고정 길이 Chunking

예를 들어 **30 token마다 무조건 자릅니다.**

In [4]:
def fixed_length_chunking(text, chunk_size=30):
    """
    문서를 일정한 token 수로 자르는 함수

    중요:
    - 실제 Chunk 크기는 원래 token ID 개수로 계산합니다.
    - decode한 문자열을 다시 tokenize하면 token 수가 달라질 수 있으므로,
      여기서는 원래 chunk_ids의 길이를 그대로 사용합니다.
    """

    # 문장을 token ID로 변환
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []

    # chunk_size만큼 반복해서 자릅니다.
    for i in range(0, len(token_ids), chunk_size):

        # 현재 chunk에 들어갈 token 선택
        # chunk_size=30이면 마지막 chunk를 제외하고 정확히 30개입니다.
        chunk_ids = token_ids[i:i + chunk_size]

        # 화면 출력을 위해 token ID를 다시 문자열로 변환
        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True
        )

        # 텍스트와 실제 token 수를 함께 저장
        chunks.append({
            "text": chunk_text,
            "token_count": len(chunk_ids)
        })

    return chunks


In [5]:
# 고정 길이 Chunking 실행
fixed_chunks = fixed_length_chunking(
    text,
    chunk_size=30
)

for i, chunk in enumerate(fixed_chunks, 1):
    print(f"[Chunk {i}]")
    print("Token 수:", chunk["token_count"])
    print(chunk["text"])
    print()


[Chunk 1]
Token 수: 30
RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다. RAG 시스

[Chunk 2]
Token 수: 30
##템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다. 이렇게 나눈 작은 문서 조각을 chunk

[Chunk 3]
Token 수: 30
##라고 합니다. Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다. 반대로 Chunk가 너무

[Chunk 4]
Token 수: 30
작으면 문장의 의미가 잘릴 수 있습니다. 따라서 문서의 특성에 맞는 Chunking 방법을 선택하는 것이

[Chunk 5]
Token 수: 30
중요합니다. 고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있습니다.

[Chunk 6]
Token 수: 25
문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.



### 관찰 포인트
고정 길이 Chunking은 문장의 의미와 관계없이 **token 수만 보고 자릅니다.**
따라서 문장 중간이 잘릴 수 있습니다.

# 방법 2. 문장 경계 Chunking

이번에는 문장을 먼저 분리한 후, 가능한 한 문장 단위로 Chunk를 만듭니다.

In [6]:
import kss

# 한국어 문장을 문장 단위로 분리
sentences = kss.split_sentences(text)

for i, sentence in enumerate(sentences, 1):
    print(i, sentence)

For your information, Kss also supports mecab backend.
We recommend you to install mecab or konlpy.tag.Mecab for faster execution of Kss.
Please refer to following web sites for details:
- mecab: https://github.com/hyunwoongko/python-mecab-kor
- konlpy.tag.Mecab: https://konlpy.org/en/latest/api/konlpy.tag/#mecab-class



1 RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다.
2 RAG 시스템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다.
3 이렇게 나눈 작은 문서 조각을 chunk라고 합니다.
4 Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다.
5 반대로 Chunk가 너무 작으면 문장의 의미가 잘릴 수 있습니다.
6 따라서 문서의 특성에 맞는 Chunking 방법을 선택하는 것이 중요합니다.
7 고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있습니다.
8 문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.


In [7]:
def sentence_chunking(text, max_tokens=30):
    """
    문장 경계를 가능한 한 유지하면서
    max_tokens 이하의 chunk를 만드는 함수
    """

    # 문장 단위로 분리
    sentences = kss.split_sentences(text)

    chunks = []

    # 현재 만들고 있는 chunk
    current_chunk = ""

    for sentence in sentences:

        # 현재 chunk에 새 문장을 추가해 봅니다.
        candidate = (
            current_chunk + " " + sentence
        ).strip()

        # 추가했을 때 token 수 계산
        token_count = len(
            tokenizer.encode(
                candidate,
                add_special_tokens=False
            )
        )

        # max_tokens 이하라면 계속 추가
        if token_count <= max_tokens:
            current_chunk = candidate

        # 너무 길어지면 현재 chunk를 저장
        else:
            if current_chunk:
                chunks.append(current_chunk)

            # 새 chunk 시작
            current_chunk = sentence

    # 마지막 chunk 저장
    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [8]:
# 문장 경계 Chunking 실행
sentence_chunks = sentence_chunking(
    text,
    max_tokens=30
)

for i, chunk in enumerate(sentence_chunks, 1):
    print(f"[Chunk {i}]")
    print(chunk)
    print()

[Chunk 1]
RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다.

[Chunk 2]
RAG 시스템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다.

[Chunk 3]
이렇게 나눈 작은 문서 조각을 chunk라고 합니다.

[Chunk 4]
Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다.

[Chunk 5]
반대로 Chunk가 너무 작으면 문장의 의미가 잘릴 수 있습니다.

[Chunk 6]
따라서 문서의 특성에 맞는 Chunking 방법을 선택하는 것이 중요합니다.

[Chunk 7]
고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있습니다.

[Chunk 8]
문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.



## 4. 두 방법을 표로 비교

In [9]:
import pandas as pd

rows = []

# 고정 길이 chunk
# token 수는 원래 chunk_ids 길이로 저장된 값을 사용합니다.
for i, chunk in enumerate(fixed_chunks, 1):

    rows.append({
        "방법": "Fixed",
        "Chunk": i,
        "Token 수": chunk["token_count"],
        "내용": chunk["text"]
    })


# 문장 기반 chunk
# 문장 기반 chunk는 문자열이므로 tokenizer로 token 수를 계산합니다.
for i, chunk in enumerate(sentence_chunks, 1):

    token_count = len(
        tokenizer.encode(
            chunk,
            add_special_tokens=False
        )
    )

    rows.append({
        "방법": "Sentence",
        "Chunk": i,
        "Token 수": token_count,
        "내용": chunk
    })


df = pd.DataFrame(rows)

df


,방법,Chunk,Token 수,내용
0,Fixed,1,30,RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다. RAG 시스
1,Fixed,2,30,##템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다. 이렇게 나눈 작은...
2,Fixed,3,30,##라고 합니다. Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다...
3,Fixed,4,30,작으면 문장의 의미가 잘릴 수 있습니다. 따라서 문서의 특성에 맞는 Chunking...
4,Fixed,5,30,중요합니다. 고정 길이 Chunking은 구현이 간단하지만 문장 중간이 잘릴 수 있...
5,Fixed,6,25,문장 기반 Chunking은 문장의 경계를 유지하기 때문에 의미를 보존하기 쉽습니다.
6,Sentence,1,26,RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용하는 기술입니다.
7,Sentence,2,23,RAG 시스템에서는 문서를 검색하기 전에 작은 단위로 나누어 저장합니다.
8,Sentence,3,15,이렇게 나눈 작은 문서 조각을 chunk라고 합니다.
9,Sentence,4,19,Chunk가 너무 크면 서로 다른 내용이 함께 들어갈 수 있습니다.


## 5. 간단한 통계 비교

In [10]:
print("=== Fixed Length Chunking ===")
print("Chunk 수:", len(fixed_chunks))

# 고정 길이 Chunking은 원래 token ID 개수를 그대로 출력합니다.
for chunk in fixed_chunks:
    print(
        chunk["token_count"],
        "tokens"
    )


print("\n=== Sentence Boundary Chunking ===")
print("Chunk 수:", len(sentence_chunks))

for chunk in sentence_chunks:
    print(
        len(
            tokenizer.encode(
                chunk,
                add_special_tokens=False
            )
        ),
        "tokens"
    )


=== Fixed Length Chunking ===
Chunk 수: 6
30 tokens
30 tokens
30 tokens
30 tokens
30 tokens
25 tokens

=== Sentence Boundary Chunking ===
Chunk 수: 8
26 tokens
23 tokens
15 tokens
19 tokens
21 tokens
21 tokens
25 tokens
25 tokens


## 정리

| 방법 | 장점 | 단점 |
|---|---|---|
| Fixed Length | 구현이 단순함, 크기가 일정함 | 문장 중간이 잘릴 수 있음 |
| Sentence Boundary | 문장의 의미가 보존됨 | Chunk 크기가 일정하지 않음 |

**핵심:** Chunking의 목적은 단순히 문서를 작게 자르는 것이 아니라, 검색에 필요한 의미 단위를 보존하면서 적절한 크기로 나누는 것입니다.